In [5]:
import geopandas as gpd
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from spreg import OLS  # pip install pysal spreg
from libpysal.weights import DistanceBand, lag_spatial, KNN
from mgwr.sel_bw import Sel_BW
from mgwr.gwr import GWR
import statsmodels.api as sm

# 1. Load and clean data

In [2]:
g  = gpd.read_file("../Data/Shapefiles/APES_metric.gpkg", layer="nuts3")
g = g.loc[(~g.geometry.is_empty) & (g["RecordType"] != "ZIKV")].copy()
g = g.dropna().reset_index(drop=True)
print(f'We have {len(g)} rows')

We have 1283 rows


In [3]:
covariates = ["Forest", "Shrub", "Urban", "Crop", "Water", "Other",
           "Mean_P_Winter", "Max_WD_Winter", "Max_DD_Winter",
           "Mean_T_Spring", "Max_DD_Spring",
           "Mean_T_Summer",
           "Max_WD_Autumn", "Max_DD_Autumn",
           "gdp", "pop_dens"]

cols_keep = (["incidence"] + covariates + ["geometry", "NUTS_NAME", "NUTS_CODE"])

In [7]:
agg_dict = {"incidence": "sum"} 
agg_dict.update({v: "mean" for v in covariates}) 
agg_dict.update({"geometry": "first", "NUTS_NAME": "first"}) 
gdf = g.groupby("NUTS_CODE", as_index=False).agg(agg_dict).set_geometry("geometry")
gdf = gpd.GeoDataFrame(gdf, geometry="geometry", crs=g.crs)
gdf = gdf.to_crs(32633)
coords = np.column_stack([gdf.geometry.centroid.x, gdf.geometry.centroid.y])
print(f'We have {len(gdf)} rows')

We have 1134 rows


# 2. Standardize covariates for the models

In [8]:
scaler = StandardScaler()
X_std = scaler.fit_transform(gdf[covariates].to_numpy())
print(f'Dimension of Matrix X of the covariates is {np.shape(X_std)}')

Dimension of Matrix X of the covariates is (1134, 16)


# 3. Run the models

In [10]:
# y raw and distances
y_raw = gdf['incidence'].to_numpy().reshape(-1, 1)  # column vector
distances = np.arange(100_000, 200_000 + 1, 10_000)  # 100–200 km

In [11]:
d = 150000
W = DistanceBand(coords, threshold=d, binary=True, silence_warnings=True)
W.transform = "R"
# --- Spatial lag then standardize y ---
y_lag = lag_spatial(W, y_raw.ravel()).reshape(-1, 1)
Wy_z = (y_lag - y_lag.mean(axis=0)) / y_lag.std(axis=0, ddof=0)

# --- OLS (global) ---
y_sm = Wy_z.ravel()                               # 1D endog for sm.OLS
X_df = pd.DataFrame(X_std, columns=covariates)
# Add intercept column
X_df = sm.add_constant(X_df)
ols_sm = sm.OLS(y_sm, X_df).fit()  # classical OLS


In [12]:
ols_sm.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.334
Model:                            OLS   Adj. R-squared:                  0.324
Method:                 Least Squares   F-statistic:                     34.96
Date:                Mon, 10 Nov 2025   Prob (F-statistic):           5.57e-87
Time:                        14:09:20   Log-Likelihood:                -1378.9
No. Observations:                1134   AIC:                             2792.
Df Residuals:                    1117   BIC:                             2877.
Df Model:                          16                                         
Covariance Type:            nonrobust                                         
=================================================================================
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const         -1.341e-16      0.024  -5.49e-15      1.000      -0.048       0.048
Forest           -0.1148      0.034     -3.375      0.001      -0.182      -0.048
Shrub             0.1962      0.028      6.917      0.000       0.141       0.252
Urban             0.0016      0.027      0.059      0.953      -0.051       0.054
Crop             -0.1252      0.034     -3.707      0.000      -0.191      -0.059
Water             0.0894      0.027      3.287      0.001       0.036       0.143
Other            -0.0043      0.030     -0.144      0.885      -0.063       0.054
Mean_P_Winter    -0.1193      0.045     -2.677      0.008      -0.207      -0.032
Max_WD_Winter     0.0522      0.053      0.991      0.322      -0.051       0.156
Max_DD_Winter    -0.1370      0.042     -3.258      0.001      -0.220      -0.054
Mean_T_Spring    -0.9179      0.123     -7.433      0.000      -1.160      -0.676
Max_DD_Spring    -0.4842      0.040    -12.226      0.000      -0.562      -0.407
Mean_T_Summer     1.5331      0.143     10.685      0.000       1.252       1.815
Max_WD_Autumn     0.0507      0.043      1.169      0.243      -0.034       0.136
Max_DD_Autumn     0.1410      0.045      3.131      0.002       0.053       0.229
gdp              -0.0677      0.033     -2.069      0.039      -0.132      -0.003
pop_dens          0.0387      0.028      1.396      0.163      -0.016       0.093
==============================================================================
Omnibus:                     1074.549   Durbin-Watson:                   0.651
Prob(Omnibus):                  0.000   Jarque-Bera (JB):            61664.038
Skew:                           4.252   Prob(JB):                         0.00
Kurtosis:                      38.111   Cond. No.                         17.5
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [13]:
bw = Sel_BW(coords, Wy_z, X_std, kernel="exponential", fixed=False).search()
gwr_fit = GWR(coords, Wy_z, X_std, bw, kernel="exponential", name_x=covariates, fixed=False, constant=True).fit()
gwr_fit.summary()

Model type                                                         Gaussian
Number of observations:                                                1134
Number of covariates:                                                    17

Global Regression Results
---------------------------------------------------------------------------
Residual sum of squares:                                            755.595
Log-likelihood:                                                   -1378.874
AIC:                                                               2791.748
AICc:                                                              2794.361
BIC:                                                              -7100.832
R2:                                                                   0.334
Adj. R2:                                                              0.324

Variable                              Est.         SE  t(Est/SE)    p-value
------------------------------- ---------- ---------- ------

In [ ]:


OLS_results = []
GWR_results = []
name_x = ['const'] + covariates  # for OLS reporting

for d in distances:
    # --- Build W; fallback if islands ---
    W = DistanceBand(coords, threshold=d, binary=True, silence_warnings=True)
    W.transform = "R"
    # --- Spatial lag then standardize y ---
    y_lag = lag_spatial(W, y_raw.ravel()).reshape(-1, 1)
    Wy_z = (y_lag - y_lag.mean(axis=0)) / y_lag.std(axis=0, ddof=0)

    # --- OLS (global) ---
    y_sm = Wy_z.ravel()                               # 1D endog for sm.OLS
    X_df = pd.DataFrame(X_std, columns=covariates)
    # Add intercept column
    X_df = sm.add_constant(X_df)
    ols_sm = sm.OLS(y_sm, X_df).fit()  # classical OLS

    



In [ ]:
ols_sm.summary()

In [ ]:
# --- GWR (local) ---
# IMPORTANT: do NOT include your own intercept when constant=True
bw = Sel_BW(coords, Wy_z, X_std, kernel="exponential", fixed=False).search()


In [ ]:
gwr_fit = GWR(coords, Wy_z, X_std, bw, kernel="exponential", name_x=covariates, fixed=False, constant=True).fit()

In [ ]:
gwr_fit.summary()

In [ ]:
import pylab as plt
import matplotlib.patches as mpatches
import cartopy.crs as ccrs
import cartopy.feature as cfeature
# --- Projection ---
proj = ccrs.PlateCarree()  # simple lat/lon projection

gdfcrs = gdf.to_crs("EPSG:4326")
url = "../Data/Shapefiles/world-administrative-boundaries.zip"
world = gpd.read_file(url)


fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={'projection': proj})
world.plot(ax=ax, color='lightgrey', edgecolor='white', linewidth=0.3, zorder=0)
gdfcrs.plot(ax=ax,  column='localR2',cmap='terrain_r', legend=True, linewidth=0.02, zorder=1, vmin=0, vmax=1,    
        legend_kwds={
        'shrink': 0.5,        # reduces height to 50%
        'aspect': 20,         # make bar thinner/wider
        'pad': 0.02,          # spacing from map
        'label': 'Local R²'   # colorbar label
    })

# --- Style ---
ax.set_axis_off()
ax.set_xlim(-10, 31)
ax.set_ylim(35, 60)


# --- Add Cartopy gridlines ---
gl = ax.gridlines(draw_labels=True, linewidth=0.3, color='gray', alpha=0.5, linestyle='--')
gl.top_labels = False
gl.right_labels = False
gl.xlabel_style = {'size': 11}
gl.ylabel_style = {'size': 11}

In [ ]:
gdf['localR2'] = gwr_fit.localR2
import pylab as plt
f,ax=plt.subplots()
gdf.plot(column='localR2',cmap='gnuplot',ax=ax, legend=True)

In [ ]:
plt.hist(gdf['localR2']);

In [ ]:
# y raw and distances
y_raw = gdf['incidence'].to_numpy().reshape(-1, 1)  # column vector
distances = np.arange(100_000, 200_000 + 1, 10_000)  # 100–200 km

OLS_results = []
GWR_results = []
name_x = ['const'] + covariates  # for OLS reporting

for d in distances:
    # --- Build W; fallback if islands ---
    W = DistanceBand(coords, threshold=d, binary=True, silence_warnings=True)
    W.transform = "R"
    # --- Spatial lag then standardize y ---
    y_lag = lag_spatial(W, y_raw.ravel()).reshape(-1, 1)
    y = (y_lag - y_lag.mean(axis=0)) / y_lag.std(axis=0, ddof=0)

    # --- OLS (global) ---
    X_ols = np.hstack([np.ones((X_std.shape[0], 1)), X_std])
    ols = OLS(y, X_ols, name_y="Wy_z", name_x=name_x, nonspat_diag=True)
    OLS_results.append({'distance': d, 'ols': ols, 'W': W})

    # --- GWR (local) ---
    # IMPORTANT: do NOT include your own intercept when constant=True
    bw = Sel_BW(coords, y, X_std, kernel="exponential", fixed=False).search()
    gwr_fit = GWR(coords, y, X_std, bw, kernel="exponential", fixed=False, constant=True).fit()
    GWR_results.append({'distance': d, 'gwr': gwr_fit, 'bw': bw, 'W': W})

    print("R2 OLS:", ols.r2)
    print("R2 Gwr", np.mean(gwr_fit.localR2))
    print(f"Distance {d/1000:.0f} km: OLS & GWR DONE (bw={bw})")

In [ ]:
import numpy as np
import geopandas as gpd
from sklearn.preprocessing import StandardScaler
from spreg import OLS
from libpysal.weights import DistanceBand, KNN, lag_spatial, lat2W
from esda.moran import Moran
from mgwr.sel_bw import Sel_BW
from mgwr.gwr import GWR

# Global X standardization (once)
scaler = StandardScaler()
Xz = scaler.fit_transform(gdf[covariates].to_numpy())
X_ols = np.hstack([np.ones((Xz.shape[0], 1)), Xz])  # intercept for OLS only
name_x = ['const'] + covariates

y_raw = gdf['incidence'].to_numpy().reshape(-1, 1)
distances = np.arange(100_000, 200_000 + 1, 10_000)

summ = []     # rows of summary metrics (OLS+GWR)
fits = {}     # store full objects if you want later

for d in distances:
    # Build W_d with fallback
    W = DistanceBand(coords, threshold=d, binary=True, silence_warnings=True)
    W.transform = "R"

    # Spatial lag then standardize globally (at this d)
    Wy = lag_spatial(W, y_raw.ravel()).reshape(-1, 1)
    Wy_z = (Wy - Wy.mean()) / Wy.std(ddof=0)

    # --- OLS ---
    ols = OLS(Wy_z, X_ols, name_y="Wy_z", name_x=name_x, nonspat_diag=True)

    # --- OLS diagnostics (version-agnostic) ---
    n = Wy_z.shape[0]
    k = X_ols.shape[1]
    
    resid = ols.u.flatten()                         # residuals
    RSS = float((resid**2).sum())
    TSS = float(((Wy_z - Wy_z.mean())**2).sum())
    
    r2 = 1.0 - RSS / TSS
    r2_adj = 1.0 - (1.0 - r2) * (n - 1) / (n - k)
    
    # Gaussian log-likelihood with ML sigma^2 = RSS/n
    sigma2 = RSS / n
    loglik = -0.5 * n * (np.log(2 * np.pi * sigma2) + 1.0)
    AIC = 2 * k - 2 * loglik
    AICc = AIC + (2 * k * (k + 1)) / (n - k - 1)

    # Moran's I on OLS residuals (use a connectivity W; KNN is robust)
    W_res = W  # reuse W; or choose a fixed KNN for residual check
    res = ols.u
    moran = Moran(res, W_res, two_tailed=False)

    # --- GWR (no manual intercept) ---
    bw = Sel_BW(coords, Wy_z, Xz, kernel="exponential", fixed=False).search()
    gwr = GWR(coords, Wy_z, Xz, bw, kernel="exponential", fixed=False, constant=True).fit()

    summ.append({
        "distance_km": d/1000,
        "OLS_AICc": AICc,
        "OLS_adjR2": r2_adj,
        "OLS_intercept": float(ols.betas[0, 0]),
        "MoranI_res": moran.I,
        "MoranI_p": moran.p_norm,
        "GWR_AICc": gwr.aicc,
        "GWR_enp": gwr.ENP,
        "GWR_localR2_median": float(np.median(gwr.localR2)),
        "GWR_bw": bw,
        "used_KNN": isinstance(W, KNN)
    })
    fits[d] = {"ols": ols, "gwr": gwr, "W": W}

# Convert 'summ' to a DataFrame to rank distances
import pandas as pd
summary_df = pd.DataFrame(summ).sort_values(["GWR_AICc","OLS_AICc"])
print(summary_df)


In [ ]:
summary_df

In [ ]:
gdf = gpd.GeoDataFrame(gdf, geometry="geometry", crs=g.crs)
gdf = gdf.to_crs(32633)
coords = np.column_stack([gdf.geometry.centroid.x, gdf.geometry.centroid.y])

scaler = StandardScaler()
X_ = scaler.fit_transform(gdf[covariates])

from spreg import OLS  # pip install pysal spreg
from libpysal.weights import DistanceBand, lag_spatial
from mgwr.sel_bw import Sel_BW
from mgwr.gwr import GWR


distances = np.arange(100_000, 200_000 + 1, 10_000)  ## Between 100 and 200 km
y_raw = gdf['incidence'].to_numpy()  # can be raw; Wy will be z-scored later

OLS_results = []
GWR_results = []

name_x = ['const'] + covariates
for d in distances:
    # 1) Row-standardized W and spatial lag (neighbor average)
    W = DistanceBand(coords, threshold=d, binary=True, silence_warnings=True)
    W.transform = "R"

    # 2) Wy and z-score AFTER lag (this is your OLS y)
    y_lag = lag_spatial(W, y_raw)
    y_lag_z = (y_lag - y_lag.mean()) / y_lag.std(ddof=0)

    
    y = y_lag_z.reshape(-1, 1)     # SAME y as OLS
    X = X_.copy()
    X = np.hstack([np.ones((X.shape[0], 1)), X])

    ols = OLS(y, X, name_y="Y_lagspatial", name_x=name_x, nonspat_diag=True)  # global diagnostics
    OLS_results.append(ols)


    bw = Sel_BW(coords, y, X, kernel="exponential", fixed=False).search()
    gwr_fit = GWR(coords, y, X, bw, kernel="exponential", fixed=False, constant=True).fit()
    GWR_results.append(gwr_fit)
    print(f'Distance for {d} DONE')